In [ ]:
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities
)
from imagematerials.vehicles.battery import ElectricVehicleBatteries, ElectricVehicleBatteries_TV, BatteryMaterials
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.electricity.battery_preprocessing import get_preprocessing_data_evbattery

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials

In [ ]:
path_base

In [ ]:
vhc_sector = get_preprocessing_data("vehicles", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir = None, 
                                    circular_economy_scenario_dirs = None)


In [ ]:
prep_data = get_preprocessing_data_evbattery(path_base=path_base, scenario = "SSP2_M_CP", year_start=1926, year_out=2100)
sec_evbattery = Sector("ev_batteries", prep_data, check_coordinates=False)

In [ ]:
prep_data.keys()

In [ ]:
sec_evbattery.coordinates.keys()

In [ ]:
time_start = 2000
time_end = 2060
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

factory = ModelFactory(
    [vhc_sector, sec_evbattery], complete_timeline
    ).add(GenericStocks, ["vehicles"]
    ).add(GenericMaterials, ["vehicles"]
    ).add(ElectricVehicleBatteries, ["ev_batteries"], input_sources={
    "stock_by_cohort": "vehicles",
    "inflow": "vehicles",
    "outflow_by_cohort": "vehicles"}
    )
model = factory.finish()

warnings.filterwarnings("ignore")
model.simulate(simulation_timeline)

list(model.ev_batteries)

In [ ]:
model.ev_batteries['stock_battery_kWh_v2g']

In [ ]:
inflow_battery = model.vehicles["inflow"].to_array() * model.ev_batteries["shares"].rename({"Cohort": "time"}) #.sel(Cohort = t)
inflow_battery

da = inflow_battery.sum("Type").sum("Region")#.drop_vars("Cohort")
da

for bat in da.BatteryType.values:
    plt.plot(da.time, da.sel(BatteryType=bat), label=str(bat))
plt.xlabel("Year")
plt.ylabel("Inflow of EV batteries (count)")
plt.legend()
plt.title("Inflow of EV batteries")

In [ ]:
model.ev_batteries["inflow_battery_kWh_v2g"]#.Type.values
# x = model.ev_batteries["energy_density"]#.to_array()#.sel(Cohort = 2000).drop_vars("Cohort")#.to_array()
# x = model.vehicles["inflow"].to_array()
# x

In [ ]:
import numpy as np
stocks = model.ev_batteries["stock_battery_materials"].to_array().sum("Type").sum("BatteryType").sum("Region")
bool(np.isclose(stocks.pint.magnitude, 0).all())



In [ ]:
inflow_bat_mat = model.ev_batteries["inflow_battery_materials"].to_array().sum("Type").sum("BatteryType").sum("Region")

for mat in inflow_bat_mat.material.values:
    plt.plot(inflow_bat_mat.time, inflow_bat_mat.sel(material=mat), label=str(mat))
plt.xlabel("Year")
plt.ylabel("Inflow of material in EV batteries (kg)")
plt.legend()
plt.title("Inflow of materials in EV batteries")

In [ ]:
inflow_bat = model.ev_batteries['stock_battery_kWh_v2g'].sum(["BatteryType","Region"])
# t="test"
for t in inflow_bat.Type.values:
    plt.plot(inflow_bat.Time, inflow_bat.sel(Type=t), label=str(t)) #
plt.xlabel("Year")
plt.ylabel("stock of EV batteries (count)")
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title("stock of EV batteries")

In [ ]:
inflow_bat

In [ ]:
inflow_bat = model.ev_batteries["inflow_battery_kWh"].to_array().sum("Type").sum("Region")

for bat in inflow_bat.BatteryType.values:
    plt.plot(inflow_bat.time, inflow_bat.sel(BatteryType=bat), label=str(bat))
plt.xlabel("Year")
plt.ylabel("Inflow of EV batteries (kWh)")
plt.legend()
plt.title("Inflow of EV batteries")

In [ ]:
inflow_vhc = model2.vehicles["inflow"].to_array().sum("Region").loc[slice(2021,2029)]

for vhc in inflow_vhc.Type.values:
    plt.plot(inflow_vhc.time, inflow_vhc.sel(Type=vhc), label=str(vhc))
plt.xlabel("Year")
plt.ylabel("Inflow of vehicles (count)")
plt.legend(bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.title("Inflow of vehicles")

In [ ]:
# print(vhc_sector.coordinates.keys()) #
print(vhc_sector.prep_data)



In [ ]:

da_plot = da.sel(Time=slice(2000,2060))
fig, ax = plt.subplots()
for b in da_plot.battery.values:
    plt.plot(da_plot.Time, da_plot.sel(battery=b), label=b)
plt.legend()

In [ ]:
vhc_sector

In [ ]:
time_start = 2020
complete_timeline = prism.Timeline(time_start, 2030, 1)
simulation_timeline = prism.Timeline(2020, 2030, 1)

factory = ModelFactory(
    vhc_sector, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(BatteryMaterials
    )
model = factory.finish()

warnings.filterwarnings("ignore")
model.simulate(simulation_timeline)

list(model.vehicles)

In [ ]:
model.battery_weights
# model.battery_materials
# model.battery_shares
# model.battery_weights.sel(Type="Medium Freight Trucks - BEV")

# Batteries